# Lab 3: Subagents through shared state

[Open in Colab](https://colab.research.google.com/github/dominicdev/test-repo-11/blob/main/labs/lab3_subagents.ipynb)

## Lab 3: Subagents through shared state (14 min)

A subagent is the **same harness with a different role prompt**. The orchestrator splits the goal and writes
task docs; workers claim a task with an atomic transaction, run, and write the result back.
They never message each other.

In [ ]:
# Setup: install the workshop package and load your Gemini API key.
import getpass, os, subprocess, sys

REPO = "git+https://github.com/dominicdev/test-repo-11.git@main"
try:
    import agent2027
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", REPO])
    import agent2027

if "GEMINI_API_KEY" not in os.environ:
    try:  # Colab: Secrets panel (key icon) -> add GEMINI_API_KEY
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API key (from AI Studio): ")

from agent2027 import config
print("model:", config.MODEL, "| search mode:", config.SEARCH_MODE)

In [ ]:
# Firestore (Firebase Spark plan, Native mode). Skip this cell to use the in-memory store.
PROJECT_ID = "your-firebase-project-id"  # <- change me
USE_FIRESTORE = False                    # <- set True once PROJECT_ID is right

if USE_FIRESTORE:
    from google.colab import auth
    auth.authenticate_user()
    config.PROJECT, config.STORE = PROJECT_ID, "firestore"
print("state store:", config.STORE)

In [ ]:
import time
from agent2027.store import get_store
from agent2027.subagents import run_system

store = get_store()

def board(run_id):
    print(f"{'id':10} {'role':13} {'status':10} {'worker':12} verified")
    for t in store.tasks(run_id):
        print(f"{t['id']:10} {t['role']:13} {t['status']:10} {str(t['worker_id']):12} {t['verified']}")

In [ ]:
GOAL = "Compare three popular vector databases for a small startup: strengths, weaknesses, pricing model."
RUN_ID = config.new_run_id()
print("run:", RUN_ID)

import threading
out = {}
t = threading.Thread(target=lambda: out.update(answer=run_system(GOAL, RUN_ID, n_workers=3)))
t.start()
while t.is_alive():       # the "task board": poll shared state while workers run
    time.sleep(5); board(RUN_ID); print()
t.join(); board(RUN_ID)
print("\n", out.get("answer"))

### Exercise: add a `fact_checker`
Results should not reach the synthesis step unchecked. Complete `review_one` so a fact checker claims a **finished,
unverified** task, checks it, and sets `verified`. The orchestrator then only uses verified results.

- **TODO 1**: claim a review with `store.claim_review(run_id, worker_id)`
- **TODO 2**: set `verified=True` if the verdict starts with `VERDICT: PASS`; otherwise `status="rejected"`

In [ ]:
from agent2027.harness import run_agent
from agent2027.subagents import ROLE_PROMPTS, _research_tools

def review_one(run_id, worker_id="fact-checker-1"):
    task = None  # TODO 1: claim a task for review
    if not task:
        return None
    verdict = run_agent(
        f"Check this result.\nSubtask: {task['prompt']}\nResult:\n{task['result']}",
        _research_tools(), ROLE_PROMPTS["fact_checker"], max_steps=8, agent_id=worker_id, run_id=run_id)
    # TODO 2: update the task: store.update_task(task["id"], verified=..., review=verdict, ...)
    return verdict

Stuck? The finished version is the `fact_checker` role in `agent2027/subagents.py`. Run the whole system with it enabled:

In [ ]:
RUN2 = config.new_run_id()
answer = run_system(GOAL, RUN2, n_workers=3, with_fact_checker=True)
board(RUN2)
print("\n", answer)

### Things to notice
- **Context isolation** is the main benefit: each worker sees only its subtask, not the whole job.
- **Shared state fits agents inside one system.** Between teams or companies you want message passing (A2A).
- **Every agent you add is another attack surface.** A worker result is *untrusted input* for the orchestrator.